In [33]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")  # project root, so `src` imports work
import pandas as pd, numpy as np
from collections import Counter
from scipy import stats
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

from src.db import get_engine
from src.eval import evaluate, per_impression_ndcg, ndcg_at_k, mrr
from src.recommenders import (random_recommender, popularity_recommender,
                              content_recommender, cf_recommender)

engine = get_engine()

In [34]:
def parse(s):
    return [(nid, int(lab)) for nid, lab in (x.split('-') for x in s.split())]

def profile_from_history(history_str):
    if not isinstance(history_str, str) or history_str.strip() == "":
        return None
    cats, subs = Counter(), Counter()
    for nid in history_str.split():
        if nid in cat_of:    cats[cat_of[nid]] += 1
        if nid in subcat_of: subs[subcat_of[nid]] += 1
    total = sum(cats.values())
    return {"cats": cats, "subs": subs, "total": total} if total else None

def hist_idxs(hstr):
    if not isinstance(hstr, str): return []
    return [iidx[n] for n in hstr.split() if n in iidx]

In [35]:
train_clicks = pd.read_sql(
    "SELECT news_id, SUM(clicked) AS clicks FROM fact_impressions GROUP BY news_id",
    engine).set_index('news_id')['clicks'].to_dict()

meta = pd.read_sql("SELECT news_id, category, subcategory FROM dim_news", engine)
cat_of    = dict(zip(meta.news_id, meta.category))
subcat_of = dict(zip(meta.news_id, meta.subcategory))

In [36]:
clicks = pd.read_sql("SELECT user_id, news_id FROM fact_impressions WHERE clicked = 1", engine)
items = clicks.news_id.unique(); users = clicks.user_id.unique()
iidx = {n: i for i, n in enumerate(items)}
uidx = {u: i for i, u in enumerate(users)}
M = csr_matrix((np.ones(len(clicks)),
               (clicks.news_id.map(iidx), clicks.user_id.map(uidx))),
               shape=(len(items), len(users)))
item_vecs = normalize(M, axis=1).tocsr()

In [ ]:
dev = pd.read_csv('../data/raw/MINDsmall_dev/behaviors.tsv', sep='\t',
       names=['impression_id','user_id','time','history','impressions'])
dev["parsed"]   = dev["impressions"].apply(parse)
dev["profile"]  = dev["history"].apply(profile_from_history)
dev["hist_idx"] = dev["history"].apply(hist_idxs)
print(dev.columns.tolist())          

['impression_id', 'user_id', 'time', 'history', 'impressions', 'parsed', 'profile', 'hist_idx']


In [38]:
random_score     = random_recommender()
popularity_score = popularity_recommender(train_clicks)
content_score    = content_recommender(cat_of, subcat_of, train_clicks)
cf_score         = cf_recommender(item_vecs, iidx, train_clicks)

scorers = {"random": random_score, "popularity": popularity_score,
           "content": content_score, "cf": cf_score}
results = {name: evaluate(fn, dev) for name, fn in scorers.items()}
pd.DataFrame(results).T

,nDCG@10,MRR
random,0.286798,0.246002
popularity,0.309817,0.267145
content,0.359417,0.321474
cf,0.302985,0.262495


In [39]:
con_arr = per_impression_ndcg(content_score, dev)
pop_arr = per_impression_ndcg(popularity_score, dev)
t, p = stats.ttest_rel(con_arr, pop_arr)
print(f"content − popularity nDCG diff = {con_arr.mean() - pop_arr.mean():.4f},  p = {p:.2e}")

content − popularity nDCG diff = 0.0496,  p = 0.00e+00


In [ ]:
dev_hist = dev[dev["profile"].notna()].reset_index(drop=True)
print(f"{len(dev_hist):,} of {len(dev):,} impressions have history ({len(dev_hist)/len(dev):.1%})")
pd.DataFrame({
    "popularity": evaluate(popularity_score, dev_hist),
    "content":    evaluate(content_score, dev_hist),
}).T

mean diff = 0.0237,  p = 2.84e-95
